In [ ]:
from pyspark.sql.functions import concat_ws, datediff, to_date, floor, col, avg, min, max, when, col, upper, substring, year
from pyspark.sql.window import Window


In [ ]:
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
display(df_pitstops.head(5))

raceId,driverId,stop,lap,time,duration,milliseconds
841,153,1,1,17:05:23,26.898,26898
841,30,1,1,17:05:52,25.021,25021
841,17,1,11,17:20:48,23.426,23426
841,4,1,12,17:22:34,23.251,23251
841,13,1,13,17:24:10,23.842,23842


## 1.1 Average time each driver spent at the pit stop for each race

I first chose the milliseconds instead of duration since the int is better to work with. Since all the data in this dataframe are in strings, I converted the milliseconds into integer. I get unique pair of raceID and DriverID, then take average of the milliseconds (with floor to round it) to get the average pitstop time, and added as a new col, then ordered them by raceID for a easier look up. Now the drivers are represented using driver ID, I will intergrate their names in later steps

In [ ]:

df = df_pitstops.withColumn("milliseconds", col("milliseconds").try_cast("int"))
driver_avg = df.groupBy("raceId", "driverId").agg(
    floor(avg("milliseconds")).alias("avg_pit_time"),
).orderBy("raceId")

display(driver_avg.head(5))

### Explanation

1. **`.withColumn("milliseconds", col("milliseconds").try_cast("int"))`** — The pit stops CSV loads all columns as strings. This converts the `milliseconds` column to an integer type so we can perform numeric aggregations. `try_cast` is used instead of `cast` so that any malformed values become `null` rather than throwing an error.

2. **`.groupBy("raceId", "driverId")`** — Groups the data by each unique combination of race and driver. Since a driver can make multiple pit stops in a single race, this groups all of a driver's pit stops within each race together.

3. **`.agg(floor(avg("milliseconds")).alias("avg_pit_time"))`** — For each race-driver group, `avg("milliseconds")` computes the mean pit stop duration across all stops that driver made in that race. `floor()` rounds the result down to the nearest whole number (since fractional milliseconds aren't meaningful), and `.alias("avg_pit_time")` names the resulting column.

4. **`.orderBy("raceId")`** — Sorts the output by `raceId` so results are organized race-by-race for easier reading.

## 1.2 Slowest and Fastest Pit Stop in Each Race

**Logic:** To find the slowest and fastest pit stop in each race, I group the pit stop data by `raceId` (so each race is one group), then use the `min` and `max` aggregation functions on the `milliseconds` column to extract the shortest and longest pit stop durations within each race. The results are ordered by `raceId` for readability.

### Explanation

1. **`min("milliseconds").alias("fastest_pit_stop in ms")`** — Within each race group, `min()` returns the smallest value in the `milliseconds` column, which corresponds to the fastest (shortest) pit stop in that race.

2. **`max("milliseconds").alias("slowest_pit_stop in ms")`** — Similarly, `max()` returns the largest value, corresponding to the slowest (longest) pit stop in that race.


In [ ]:
race_stats = df.groupBy("raceId").agg(
    min("milliseconds").alias("fastest_pit_stop in ms"),
    max("milliseconds").alias("slowest_pit_stop in ms"),
).orderBy("raceId")

display(race_stats.head(5))

raceId,fastest_pit_stop in ms,slowest_pit_stop in ms
1000,21291,25126
1001,18084,35451
1002,23728,38563
1003,24340,34022
1004,29266,37446


In [ ]:
drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)

### Explanation

1. **`drivers.select("driverId", "forename", "surname")`** — Selects only the columns we need from the `drivers` dataset to avoid bringing in unnecessary columns.

2. **`.join(..., on="driverId")`** — Performs an inner join between `driver_avg` and the selected driver columns, matching rows where `driverId` is the same in both dataframes.


## Drivers Pit Stop Average with Driver Names

**Logic:** The `driver_avg` dataframe from 1.1 only identifies drivers by `driverId`. To make results human-readable, I join it with the `drivers` dataset on `driverId` to bring in each driver's first name (`forename`) and last name (`surname`).

In [ ]:


drivers_pitstops = driver_avg.join(
    drivers.select("driverId", "forename", "surname"),
    on="driverId"
).orderBy("raceId")


display(drivers_pitstops.head(5))

driverId,raceId,avg_pit_time,forename,surname
842,1000,21684,Pierre,Gasly
815,1000,22561,Sergio,Pérez
154,1000,21733,Romain,Grosjean
20,1000,23111,Sebastian,Vettel
839,1000,22258,Esteban,Ocon


## 2. Rank order by finishing position the average time spent at the pit stop in each race

### Explanation

1. **`driver_standing.select("raceId", "driverId", "position")`** — Selects only the relevant columns from the standings dataset to keep the join clean.

2. **`.join(..., on=["raceId", "driverId"])`** — Joins on both `raceId` and `driverId` to ensure we match each driver's pit stop average to their finishing position in the correct race (not across different races).

3. **`.withColumn("position", col("position").cast("int"))`** — Converts the `position` column from string to integer so that sorting works numerically (e.g., position 2 comes before position 10, not after).

4. **`.orderBy("position", "avg_pit_time")`** — Sorts primarily by finishing position (1st, 2nd, 3rd, etc.), and within the same position, by average pit stop time. This reveals whether faster pit stops correlate with better finishing positions.

In [ ]:
driver_standing = spark.read.csv("/Volumes/gr5069/raw/f1_data/driver_standings.csv", header=True)
display(driver_standing.head(5))

driverStandingsId,raceId,driverId,points,position,positionText,wins
1,18,1,10,1,1,1
2,18,2,8,2,2,0
3,18,3,6,3,3,0
4,18,4,5,4,4,0
5,18,5,4,5,5,0


**Logic:** To rank drivers by finishing position alongside their average pit stop time, I join the `drivers_pitstops` dataframe (which contains each driver's average pit stop time per race) with the `driver_standings` dataset on both `raceId` and `driverId`. This brings in each driver's finishing `position` for that race. I then cast the `position` column to integer (since it loads as a string from CSV) and sort the result first by finishing position, then by average pit stop time within each position. 

In [ ]:
pitstop_position = drivers_pitstops.join(
    driver_standing.select("raceId", "driverId", "position"),
    on=["raceId", "driverId"]
).withColumn("position", col("position").cast("int")).orderBy( "position", "avg_pit_time")

display(pitstop_position.head(5))


raceId,driverId,avg_pit_time,forename,surname,position
869,4,17756,Fernando,Alonso,1
1076,844,17973,Charles,Leclerc,1
1088,830,18253,Max,Verstappen,1
959,1,18543,Lewis,Hamilton,1
1064,830,18860,Max,Verstappen,1



## 3. Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

**Logic:** Some drivers in the `drivers` dataset have a missing `code` value, represented as `\N`. To fill these in, I generate a code from the first three letters of the driver's surname, converted to uppercase (e.g., "Alonso" → "ALO"). This mirrors the convention used for existing codes in the dataset. I use a conditional check: if the code is `\N`, replace it with the generated code; otherwise, keep the existing code.

### Explanation

1. **`when(col("code") == "\\N", ...)`** — The `when` function acts as a conditional (like an if-statement). It checks whether the `code` column equals `\N`, which is how missing values are represented in this CSV dataset.

2. **`upper(substring(col("surname"), 1, 3))`** — When the code is missing, this generates a replacement: `substring(col("surname"), 1, 3)` extracts the first 3 characters of the driver's surname, and `upper()` converts them to uppercase to match the format of existing codes (e.g., "Alonso" → "ALO").

3. **`.otherwise(col("code"))`** — If the code is not `\N` (i.e., it already exists), the original code value is kept unchanged.

4. **`.withColumn("code", ...)`** — Overwrites the existing `code` column with the result of the conditional expression, so the dataframe now has no missing codes.

5. The final join with `pitstop_position` on `driverId` brings the fixed codes and `dob` into the main working dataframe for use in later questions.

In [ ]:
drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
display(drivers.head(5))

driverId,driverRef,number,code,forename,surname,dob,nationality,url
1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton
2,heidfeld,\N,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld
3,rosberg,6,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg
4,alonso,14,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso
5,kovalainen,\N,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen


In [ ]:
df_drivers_fixed = drivers.withColumn(

    "code",
    when(col("code") == "\\N", upper(substring(col("surname"), 1, 3)))
    .otherwise(col("code"))
)

In [ ]:
display(df_drivers_fixed.head(5))

driverId,driverRef,number,code,forename,surname,dob,nationality,url
1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton
2,heidfeld,\N,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld
3,rosberg,6,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg
4,alonso,14,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso
5,kovalainen,\N,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen


In [ ]:
pit_avg_by_finish_pos = pitstop_position.join(
    df_drivers_fixed.select("driverId","code", "dob"),
    on="driverId"
).orderBy("driverId")
display(pit_avg_by_finish_pos.head(5))

driverId,raceId,avg_pit_time,forename,surname,position,code,dob
1,851,19866,Lewis,Hamilton,3,HAM,1985-01-07
1,843,20659,Lewis,Hamilton,2,HAM,1985-01-07
1,848,20529,Lewis,Hamilton,4,HAM,1985-01-07
1,872,20736,Lewis,Hamilton,2,HAM,1985-01-07
1,875,19954,Lewis,Hamilton,4,HAM,1985-01-07


## 4. Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

**Logic:** "Age" is defined as the driver's age **at the time of each race**, not their current age. This is calculated by finding the difference in days between the race date and the driver's date of birth (`dob`), then dividing by 365 and rounding down with `floor`. This gives an integer age that reflects how old the driver was on race day.

To find the youngest and oldest driver in each race, I group by `raceId`, compute the minimum and maximum age, then join back to the main dataframe to retrieve the driver names corresponding to those ages.

In [ ]:
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
display(df_races.head(5))

raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
1,2009,1,1,Australian Grand Prix,2009-03-29,06:00:00,http://en.wikipedia.org/wiki/2009_Australian_Grand_Prix,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
2,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Grand_Prix,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
3,2009,3,17,Chinese Grand Prix,2009-04-19,07:00:00,http://en.wikipedia.org/wiki/2009_Chinese_Grand_Prix,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
4,2009,4,3,Bahrain Grand Prix,2009-04-26,12:00:00,http://en.wikipedia.org/wiki/2009_Bahrain_Grand_Prix,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
5,2009,5,4,Spanish Grand Prix,2009-05-10,12:00:00,http://en.wikipedia.org/wiki/2009_Spanish_Grand_Prix,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N


### Explanation

**Computing Age (cells above):**

1. **`.join(df_races.select("raceId", col("date").alias("race_date")), on="raceId")`** — Joins the races dataset to bring in each race's date. The column is aliased to `race_date` to avoid confusion with other date columns.

2. **`datediff(to_date(col("race_date")), to_date(col("dob")))`** — `to_date()` converts the string columns to proper date types, and `datediff()` computes the number of days between the race date and the driver's date of birth.

3. **`floor(... / 365).cast("int")`** — Divides the day difference by 365 to get approximate years, then `floor()` rounds down (so a driver who is 25 years and 11 months old is reported as 25). The result is cast to integer for clean output.

**Finding Youngest and Oldest:**

4. **`.groupBy("raceId").agg(min("age").alias("min_age"))`** — For each race, finds the minimum age across all drivers. A separate aggregation finds the maximum age.

5. **`.join(youngest, on="raceId").filter(col("age") == col("min_age"))`** — Joins the min-age result back to the main dataframe and filters to keep only the row(s) where the driver's age matches the minimum. This retrieves the actual driver name, not just the age value.

6. **`concat_ws(" ", "forename", "surname").alias("youngest_driver")`** — Concatenates the first and last name with a space separator to create a readable full name.

7. **`youngest_drivers.join(oldest_drivers, on="raceId")`** — Combines the youngest and oldest results into a single table, with one row per race showing both the youngest and oldest driver.

**Logic:** To count cumulative podiums for each driver at any given race, I use the `results` dataset (which has the actual race finishing position, unlike `driver_standings`). I filter out non-finishers (where position is `\N`), then use a window function partitioned by `driverId` and ordered by `raceId`. For each row, I compute a running sum of wins (position == 1), 2nd places (position == 2), and 3rd places (position == 3) from the first race up to and including the current race. This gives a cumulative count that grows over time — at any given race, you can see how many podiums that driver has accumulated in their career up to that point.

concat the race date to the df

In [ ]:
pit_avg_by_finish_pos_with_age = pit_avg_by_finish_pos.join(
    df_races.select("raceId", col("date").alias("race_date")),
    on="raceId"
)

In [ ]:

pit_avg_by_finish_pos = pit_avg_by_finish_pos_with_age.withColumn(
    "age",
    floor(datediff(to_date(col("race_date")), to_date(col("dob"))) / 365).cast("int")
)
display(pit_avg_by_finish_pos.head(5))

raceId,driverId,avg_pit_time,forename,surname,position,code,dob,race_date,age
841,153,25903,Jaime,Alguersuari,11,ALG,1990-03-23,2011-03-27,21
841,22,26175,Rubens,Barrichello,16,BAR,1972-05-23,2011-03-27,38
843,1,20659,Lewis,Hamilton,2,HAM,1985-01-07,2011-04-17,26
843,22,21595,Rubens,Barrichello,17,BAR,1972-05-23,2011-04-17,38
843,2,22018,Nick,Heidfeld,8,HEI,1977-05-10,2011-04-17,33


the youngest and the oldest driver in each race, filter and concat

### Explanation

1. **`.filter(col("position") != "\\N")`** — Removes rows where the driver did not finish the race (position is `\N`), since non-finishers don't have a valid finishing position.

2. **`.withColumn("position", col("position").cast("int"))`** — Casts the position from string to integer so numeric comparisons (== 1, == 2, == 3) work correctly.

3. **`Window.partitionBy("driverId").orderBy("raceId").rowsBetween(Window.unboundedPreceding, Window.currentRow)`** — Defines a window frame for each driver, ordered chronologically by `raceId`. The frame spans from the driver's very first race (`unboundedPreceding`) up to the current row (`currentRow`), enabling cumulative (running) aggregations.

4. **`sum(when(col("position") == 1, 1).otherwise(0)).over(w)`** — For each row, `when` returns 1 if the driver won that race and 0 otherwise. `sum(...).over(w)` computes the running total across the window, giving the cumulative number of wins up to that race. The same pattern is repeated for 2nd and 3rd place finishes.

5. **`podiums_slim = podiums.select("raceId", "driverId", "num_wins", "num_2nd", "num_3rd")`** — Selects only the columns needed for the join, keeping the dataframe lean.

6. **`.join(podiums_slim, on=["raceId", "driverId"], how="left")`** — Left-joins the podium counts back into the main working dataframe. A left join is used so that drivers who appear in the pit stop data but not in results are retained with null podium counts rather than being dropped.

In [ ]:
# find the minimum age 
youngest = pit_avg_by_finish_pos.groupBy("raceId").agg(
    min("age").alias("min_age")
)

# find the maximum age
oldest = pit_avg_by_finish_pos.groupBy("raceId").agg(
    max("age").alias("max_age")
)

# match youngest age back to get driver name
youngest_drivers = pit_avg_by_finish_pos.join(youngest, on="raceId") \
    .filter(col("age") == col("min_age")) \
    .select("raceId", concat_ws(" ", "forename", "surname").alias("youngest_driver"), col("age").alias("youngest_age"))

# match oldest age back to get driver name
oldest_drivers = pit_avg_by_finish_pos.join(oldest, on="raceId") \
    .filter(col("age") == col("max_age")) \
    .select("raceId", concat_ws(" ", "forename", "surname").alias("oldest_driver"), col("age").alias("oldest_age"))

# combine the resuls into one table 
result = youngest_drivers.join(oldest_drivers, on="raceId").orderBy("raceId")
display(result.head(5))


raceId,youngest_driver,youngest_age,oldest_driver,oldest_age
1000,Lance Stroll,19,Kimi Räikkönen,38
1001,Lance Stroll,19,Kimi Räikkönen,38
1002,Lance Stroll,19,Kimi Räikkönen,38
1003,Lance Stroll,19,Kimi Räikkönen,38
1004,Lance Stroll,19,Kimi Räikkönen,38


## 5. At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

### Explanation

1. **`year(to_date(col("race_date")))`** — Converts the `race_date` string to a date type with `to_date()`, then extracts just the year component using `year()`. This new column `race_year` allows us to group data by season.

2. **`.groupBy("race_year", "driverId", "forename", "surname", "code")`** — Groups the data by year and driver. Including the name and code columns in the group-by ensures they are carried through to the output without needing a separate join.

3. **`round(avg("avg_pit_time"), 0).alias("yearly_avg_pit_time")`** — Computes the mean of each driver's per-race average pit stop times within each year, rounded to the nearest whole millisecond. This gives one number per driver per season summarizing their pit stop performance.

4. **`.orderBy("driverId", "race_year")`** — Sorts by driver and then by year, so you can trace each driver's pit stop trend over their career.

In [ ]:
display(pit_avg_by_finish_pos.head(5))

raceId,driverId,avg_pit_time,forename,surname,position,code,dob,race_date,age
841,153,25903,Jaime,Alguersuari,11,ALG,1990-03-23,2011-03-27,21
841,22,26175,Rubens,Barrichello,16,BAR,1972-05-23,2011-03-27,38
843,1,20659,Lewis,Hamilton,2,HAM,1985-01-07,2011-04-17,26
843,22,21595,Rubens,Barrichello,17,BAR,1972-05-23,2011-04-17,38
843,2,22018,Nick,Heidfeld,8,HEI,1977-05-10,2011-04-17,33


In [ ]:
race_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)

### Explanation

1. **`.filter(col("race_year") == 2021)`** — Filters the dataframe to keep only rows from the 2021 season.

2. **`.orderBy("race_date", "avg_pit_time")`** — Sorts first by race date (chronologically through the season) and then by pit stop time within each race, making it easy to spot which specific races had unusually long pit stops.

**Findings:** The year 2021 shows a mix of normal and extremely long pit stops. The outliers are likely caused by incidents during races (e.g., red flags, safety cars, or mechanical issues) where drivers were stuck in the pit lane for extended periods, inflating the average.

In [ ]:
display(race_results.head(5))

resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
1,18,1,1,22,1,1,1,1,10,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
2,18,2,2,3,5,2,2,2,8,58,+5.478,5696094,41,3,1:27.739,217.586,1
3,18,3,3,7,7,3,3,3,6,58,+8.163,5698779,41,5,1:28.090,216.719,1
4,18,4,4,5,11,4,4,4,5,58,+17.181,5707797,58,7,1:28.603,215.464,1
5,18,5,1,23,3,5,5,5,4,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [ ]:

# cast position, filter out non-finishers
res = race_results.filter(col("position") != "\\N") \
    .withColumn("position", col("position").cast("int"))

# cumulative podium count per driver up to each race
w = Window.partitionBy("driverId").orderBy("raceId").rowsBetween(Window.unboundedPreceding, Window.currentRow)

podiums = res.withColumn(
    "num_wins", sum(when(col("position") == 1, 1).otherwise(0)).over(w)
).withColumn(
    "num_2nd", sum(when(col("position") == 2, 1).otherwise(0)).over(w)
).withColumn(
    "num_3rd", sum(when(col("position") == 3, 1).otherwise(0)).over(w)
)

display(podiums.select("raceId", "driverId", "position", "num_wins", "num_2nd", "num_3rd").orderBy("driverId", "raceId").head(5))


raceId,driverId,position,num_wins,num_2nd,num_3rd
10,1,1,1,0,0
1000,1,1,2,0,0
1001,1,2,2,1,0
1002,1,1,3,1,0
1003,1,1,4,1,0


In [ ]:
podiums_slim = podiums.select("raceId", "driverId", "num_wins", "num_2nd", "num_3rd")

pit_avg_by_finish_pos = pit_avg_by_finish_pos.join(podiums_slim, on=["raceId", "driverId"], how="left")

display(pit_avg_by_finish_pos.head(5))


raceId,driverId,avg_pit_time,forename,surname,position,code,dob,race_date,age,num_wins,num_2nd,num_3rd
841,153,25903,Jaime,Alguersuari,11,ALG,1990-03-23,2011-03-27,21,0,0,0
841,22,26175,Rubens,Barrichello,16,BAR,1972-05-23,2011-03-27,38,null,null,null
843,1,20659,Lewis,Hamilton,2,HAM,1985-01-07,2011-04-17,26,54,37,24
843,22,21595,Rubens,Barrichello,17,BAR,1972-05-23,2011-04-17,38,11,23,26
843,2,22018,Nick,Heidfeld,8,HEI,1977-05-10,2011-04-17,33,0,8,5


## 6. Continue exploring the data by answering your own question.

**Question: Does the average pit stop time improve (get shorter) over the years?**

**Logic:** To investigate this, I extract the year from each race's date and then group by year (along with driver info) to compute the average pit stop time per driver per year. By looking at the trend across years, we can see whether pit stop efficiency has generally improved over time, or if there are anomalies in certain years.

In [ ]:
# extract year from race_date
pit_by_year = pit_avg_by_finish_pos.withColumn("race_year", year(to_date(col("race_date"))))

# average pit time per driver per year
yearly_avg = pit_by_year.groupBy("race_year", "driverId", "forename", "surname", "code").agg(
    round(avg("avg_pit_time"), 0).alias("yearly_avg_pit_time")
).orderBy("driverId", "race_year")

display(yearly_avg.head(5))

race_year,driverId,forename,surname,code,yearly_avg_pit_time
2011,1,Lewis,Hamilton,HAM,22423.0
2012,1,Lewis,Hamilton,HAM,22778.0
2013,1,Lewis,Hamilton,HAM,22898.0
2014,1,Lewis,Hamilton,HAM,45401.0
2015,1,Lewis,Hamilton,HAM,24150.0


It seems like the calculation for the duration of a pit stop have changed in year 2016, 2020-2024, since the time is significantly longer than the other years. 2021 is the slowest avg pit stop for all drivers 

**Further Investigation:** To understand why 2021 had anomalously high average pit stop times, I filter the data to only that year and order by race date and pit time to inspect individual races.

In [ ]:
pit_2021 = pit_by_year.filter(col("race_year") == 2021)

display(pit_2021.orderBy("race_date", "avg_pit_time").head(5))


raceId,driverId,avg_pit_time,forename,surname,position,code,dob,race_date,age,num_wins,num_2nd,num_3rd,race_year
1052,8,24076,Kimi,Räikkönen,11,RAI,1979-10-17,2021-03-28,41,2,3,4,2021
1052,815,24096,Sergio,Pérez,5,PER,1990-01-26,2021-03-28,31,1,1,0,2021
1052,830,24307,Max,Verstappen,2,VER,1997-09-30,2021-03-28,23,6,13,10,2021
1052,832,24347,Carlos,Sainz,8,SAI,1994-09-01,2021-03-28,26,0,1,1,2021
1052,847,24434,George,Russell,14,RUS,1998-02-15,2021-03-28,23,0,0,0,2021


The year 2021 has a mix of seemingly normal pitstops and extemely long pitstop. My guess the accidents might contributed to the longer than average pitstops